In [ ]:
"""
bench_v2.py — Complete JLI benchmark suite
Run:  python bench_v2.py
Output: bench_results/bench_[A,B,C,2,6,8,D,E].csv

Bench A  — Competitive vs SkipList  (hops + latency + throughput)
Bench B  — Structural Invariants    (pointer/junction validity only)
Bench B2/B3 - measure the position of the junction and the shortcuts within the segment (They can show the issue but that how structure design for the drift)
Bench C  — Workload Independence    (oscillation + bounded ceiling)
Bench 2  — N-scaling (5k→200k)
Bench 6  — Config quality
Bench 8  — Long-run latency drift (300k ops)
Bench D  — c*√N constant stability  (does c stay fixed across N?)
Bench E  — Shortcut scaling trend   (what is k* = f(seg_size)?)
"""
# ══════════════════════════════════════════════════════════════════════════════
# JLI IMPLEMENTATION
# ══════════════════════════════════════════════════════════════════════════════

class Node:
    __slots__ = ("value", "next", "payload")
    def __init__(self, value, payload=None):
        self.value   = value
        self.next    = None
        self.payload = payload
    def __repr__(self): return "Node(%d)" % self.value


class Junction:
    __slots__ = ("node","segment_start","segment_end","prev_junction",
                 "next_junction","shortcuts","segment_len","runtime_len")
    def __init__(self, node):
        self.node = node; self.segment_start = node; self.segment_end = node
        self.prev_junction = None; self.next_junction = None
        self.shortcuts = []; self.segment_len = 0; self.runtime_len = 0


def _segment_len_hint(actual_len, segment_size, tol=0.02):
    if abs(actual_len - segment_size) <= segment_size * tol:
        return segment_size
    return actual_len


def _plan_shortcuts_offsets(seg_len, K):
    needed = max(0, max(2, int(K)) - 2)
    if seg_len <= 1 or needed <= 0: return []
    mid_idx = (seg_len - 1) // 2
    intervals = [(0, mid_idx), (mid_idx, seg_len - 1)]
    offsets = []
    while needed > 0 and intervals:
        l, r = intervals.pop(0)
        if r - l <= 1: continue
        mid = (l + r) // 2
        if mid not in (0, mid_idx, seg_len - 1):
            offsets.append(mid); needed -= 1
        if needed > 0:
            intervals.append((l, mid)); intervals.append((mid, r))
    return offsets


def _plan_segment(seg_len, K):
    if seg_len <= 1:
        return {"seg_len": seg_len, "junction_offset": 0, "shortcut_offsets": []}
    return {"seg_len": seg_len, "junction_offset": (seg_len - 1) // 2,
            "shortcut_offsets": _plan_shortcuts_offsets(seg_len, K)}


def _compute_segment_ranges(n, segment_size, tol=0.02):
    size = max(1, segment_size); ranges = []; start = 1
    while start <= n:
        end = min(start + size - 1, n); actual = end - start + 1
        ranges.append((start, end, _segment_len_hint(actual, size, tol=tol)))
        start = end + 1
    return ranges


def _is_live(j, jid_set):
    return (j is not None and id(j) in jid_set
            and j.segment_start is not None
            and j.segment_end is not None and j.node is not None)


class _JLIEngine:
    def __init__(self, segment_size=100, shortcuts_per_junction=3,
                 enable_rebuild=True, local_interval=1000, t_j=0.15,
                 sub_interval=5000, soft_pct=0.25, hard_pct=0.50,
                 flagged_ratio_limit=0.30, min_seg_len_pct=0.25,
                 max_suboptimal_segments=0.3,
                 min_suboptimal_events_before_global=5,
                 emergency_hard_segment_ratio=0.95, stop_crash_local_sub=0.08):
        self.head = None; self.length = 0
        self.junctions = []; self._jid_set = set()
        self.segment_size = max(1, segment_size)
        self.K = max(2, shortcuts_per_junction)
        self.enable_rebuild = enable_rebuild
        self.local_interval = local_interval; self.t_j = t_j
        self.sub_interval = sub_interval; self.soft_pct = soft_pct
        self.hard_pct = hard_pct; self.flagged_ratio_limit = flagged_ratio_limit
        self.min_seg_len_pct = min_seg_len_pct
        self.max_suboptimal_segments = max_suboptimal_segments
        self.min_suboptimal_events_before_global = min_suboptimal_events_before_global
        self.emergency_hard_segment_ratio = emergency_hard_segment_ratio
        self.stop_crash_local_sub = stop_crash_local_sub
        self._last_junction = None; self._last_prev_junction = None
        self.local_scan_counter = 0; self.suboptimal_scan_count = 0
        self.local_rebuild_events = 0; self.suboptimal_rebuild_events = 0
        self.global_rebuild_events = 0; self.local_rebuild_cost = 0
        self.suboptimal_segments_cost = 0; self.global_rebuild_cost = 0
        self.local_ops_counter = 0; self.sub_ops_counter = 0
        self.sub_event_before_global_rebuild = 0
        self.search_count = 0; self.insert_count = 0; self.delete_count = 0

    def _refresh_jid(self):
        self._jid_set = {id(j) for j in self.junctions if j is not None}
    def _live(self, j): return _is_live(j, self._jid_set)
    def _ps(self, seg_len): return _plan_segment(seg_len, self.K)
    def _dist(self, a, b):
        d = 1; cur = a
        while cur and cur is not b: cur = cur.next; d += 1
        return d
    def size(self): return self.length
    def get_metrics(self):
        return {"local_scan": self.local_scan_counter,
                "sub_scan":   self.suboptimal_scan_count,
                "local_event":  self.local_rebuild_events,
                "sub_event":    self.suboptimal_rebuild_events,
                "global_event": self.global_rebuild_events,
                "local_cost":   self.local_rebuild_cost,
                "sub_cost":     self.suboptimal_segments_cost,
                "global_cost":  self.global_rebuild_cost}

    def _build_junctions(self):
        self.junctions = []
        if not self.head: self._refresh_jid(); return
        ranges = _compute_segment_ranges(self.length, self.segment_size, tol=0.02)
        plans  = [_plan_segment(sl, self.K) for _, _, sl in ranges]
        cur    = self.head; seg_i = seg_offset = 0
        seg_start = jn = None; shortcuts = []; sc_set = set()
        while cur and seg_i < len(plans):
            plan = plans[seg_i]; sl = plan["seg_len"]
            if seg_offset == 0:
                seg_start = cur; jn = None; shortcuts = [cur]
                sc_set = set(plan["shortcut_offsets"])
            if seg_offset == plan["junction_offset"]: jn = cur
            if seg_offset in sc_set: shortcuts.append(cur)
            if seg_offset == sl - 1:
                seg_end = cur; shortcuts.append(cur)
                if len(shortcuts) < self.K:
                    shortcuts.extend([seg_start] * (self.K - len(shortcuts)))
                if jn is None: jn = seg_start
                j = Junction(jn); j.segment_start = seg_start
                j.segment_end = seg_end; j.segment_len = sl; j.shortcuts = shortcuts
                self.junctions.append(j)
                seg_i += 1; seg_offset = -1; shortcuts = []; jn = None; sc_set = set()
            cur = cur.next; seg_offset += 1
        n = len(self.junctions)
        for i in range(n):
            self.junctions[i].prev_junction = self.junctions[i - 1]
            self.junctions[i].next_junction = self.junctions[(i + 1) % n]
        self._refresh_jid()

    def build_from_values(self, values, key=lambda x: x):
        self.head = None; prev = None; self.length = 0
        for item in values:
            v = key(item); node = Node(v, payload=item)
            if self.head is None: self.head = node
            else: prev.next = node
            prev = node; self.length += 1
        self._build_junctions()

    def _choose_start(self, target):
        if not self.junctions: return None
        cands = []
        f = self.junctions[0]; la = self.junctions[-1]
        if self._live(f): cands.append(f)
        if self._live(la) and la is not f: cands.append(la)
        if self._live(self._last_junction): cands.append(self._last_junction)
        if self._live(self._last_prev_junction): cands.append(self._last_prev_junction)
        if not cands: return None
        for j in cands:
            if j.segment_start.value <= target <= j.segment_end.value: return j
        return min(cands, key=lambda j: min(abs(target - j.segment_start.value),
                                             abs(target - j.segment_end.value)))

    def search(self, target):
        self.search_count += 1
        if not self.junctions: return None
        cj = self._choose_start(target); visited = set()
        while cj not in visited:
            if not self._live(cj): return None
            visited.add(cj)
            ss = cj.segment_start.value; se = cj.segment_end.value
            if not (ss <= target <= se):
                cj = cj.prev_junction if target < ss else cj.next_junction; continue
            sn = (cj.segment_start if (cj.node and target < cj.node.value)
                  else (cj.node or cj.segment_start))
            valid = [sc for sc in cj.shortcuts if sn.value < sc.value <= target]
            if valid:
                best = max(valid, key=lambda n: n.value)
                if best.value > sn.value: sn = best
            cur = sn
            while cur and cur.value <= se:
                if cur.value == target: self._last_junction = cj; return cur
                if cur.value > target: return None
                cur = cur.next
            return None
        return None

    def _linear_scan(self, target):
        prev = None; cur = self.head
        while cur and cur.value < target: prev = cur; cur = cur.next
        return prev, cur, None

    def mutation_search(self, target):
        if not self.junctions: return self._linear_scan(target)
        cj = self._choose_start(target)
        if not cj or not self._live(cj): return self._linear_scan(target)
        prev_junc = None; visited = set(); direction = None
        while cj not in visited:
            if not self._live(cj): return self._linear_scan(target)
            visited.add(cj)
            ss = cj.segment_start.value; se = cj.segment_end.value
            if ss <= target <= se: break
            prev_junc = cj
            if target < ss: direction = "r2l"; cj = cj.prev_junction
            else:           direction = "l2r"; cj = cj.next_junction
        else:
            if target < self.junctions[0].segment_start.value:
                self._last_prev_junction = None
                self._last_junction = self.junctions[0]
                return None, self.junctions[0].segment_start, self.junctions[0]
            else:
                self._last_prev_junction = (self.junctions[-2]
                                            if len(self.junctions) > 1 else None)
                self._last_junction = self.junctions[-1]
                return self.junctions[-1].segment_end, None, self.junctions[-1]
        self._last_prev_junction = prev_junc; self._last_junction = cj
        if target == cj.segment_start.value:
            if cj.segment_start is self.head:
                self._last_junction = cj; return None, self.head, cj
            fb = (cj.prev_junction if direction in ("l2r", None) else cj.next_junction)
            if fb and self._live(fb):
                anchor = fb.shortcuts[-1] if fb.shortcuts else fb.segment_end
                sp = anchor; sn = anchor.next
            else: sp = None; sn = self.head
            prev = sp; cur = sn
            while cur and cur.value < target: prev = cur; cur = cur.next
            self._last_junction = cj; return prev, cur, cj
        sn = cj.segment_start; sp = None
        if cj.node and target > cj.node.value: sn = cj.node
        psc = None
        for sc in cj.shortcuts:
            if sc.value < target: psc = sc
            else: break
        if psc: sn = psc
        if (self._last_prev_junction and self._live(self._last_prev_junction)
                and cj.segment_start is not self.head
                and target != cj.segment_start.value):
            cp = self._last_prev_junction.segment_end
            cn = cp.next if cp else None
            if cn and cn.value <= target: sp = cp; sn = cn
        prev = sp; cur = sn
        while cur and cur.value < target: prev = cur; cur = cur.next
        self._last_junction = cj; return prev, cur, cj

    def insert(self, value, payload=None, allow_duplicate=False):
        nn = Node(value, payload)
        if not self.head:
            self.head = nn; self.length = 1; self._build_junctions()
            self.insert_count += 1; return True
        prev, cur, j = self.mutation_search(value)
        if cur and cur.value == value and not allow_duplicate: return False
        nn.next = cur
        if prev: prev.next = nn
        else: self.head = nn
        self.length += 1
        if j is not None:
            if prev is None or value < j.segment_start.value:
                j.segment_start = nn
                if j.shortcuts and j.shortcuts[0] is cur: j.shortcuts[0] = nn
            elif prev is j.segment_end:
                j.segment_end = nn
                if j.shortcuts: j.shortcuts[-1] = nn
        self.local_ops_counter += 1; self.sub_ops_counter += 1
        if self.enable_rebuild: self._maintenance_hook()
        self.insert_count += 1; return True

    def delete(self, value):
        prev, cur, j = self.mutation_search(value)
        if not cur or cur.value != value: return False
        nxt = cur.next
        if prev: prev.next = nxt
        else: self.head = nxt
        self.length -= 1
        if self.length == 0:
            self.junctions = []; self._refresh_jid()
            self._last_junction = self._last_prev_junction = None
            self.delete_count += 1; return True
        if j is not None:
            nd = cur
            if nd is j.segment_start:
                j.segment_start = nxt
                for i in range(len(j.shortcuts)):
                    if j.shortcuts[i] is nd: j.shortcuts[i] = nxt
            elif nd is j.segment_end:
                j.segment_end = prev
                if j.shortcuts: j.shortcuts[-1] = prev
            elif nd is j.node:
                slow = fast = j.segment_start
                while (fast and fast.next and fast is not j.segment_end
                       and fast.next is not j.segment_end):
                    fast = fast.next.next; slow = slow.next
                j.node = slow
            elif nd in j.shortcuts:
                idx   = j.shortcuts.index(nd)
                left  = j.shortcuts[idx - 1] if idx > 0 else j.segment_start
                right = (j.shortcuts[idx + 1] if idx + 1 < len(j.shortcuts)
                         else j.segment_end)
                if left and right and left is not right:
                    slow = fast = left
                    while (fast and fast.next and fast is not right
                           and fast.next is not right):
                        fast = fast.next.next; slow = slow.next
                    j.shortcuts[idx] = slow
                else: j.shortcuts[idx] = left or right or j.segment_start
            sl = self._dist(j.segment_start, j.segment_end)
            if sl <= 2:
                vals = []; c2 = j.segment_start
                for _ in range(sl):
                    if c2 is None: break
                    vals.append(c2.value); c2 = c2.next
                if j in self.junctions: self.junctions.remove(j)
                n2 = len(self.junctions)
                for i in range(n2):
                    self.junctions[i].prev_junction = self.junctions[i - 1]
                    self.junctions[i].next_junction = self.junctions[(i + 1) % n2]
                self._refresh_jid()
                self._last_junction = self._last_prev_junction = None
                old_l = self.local_ops_counter; old_s = self.sub_ops_counter
                old_en = self.enable_rebuild; self.enable_rebuild = False
                for v in vals: self.insert(v)
                self.enable_rebuild = old_en
                self.local_ops_counter = old_l; self.sub_ops_counter = old_s
                self._last_junction = self._last_prev_junction = None
                self.delete_count += 1; return True
        self.local_ops_counter += 1; self.sub_ops_counter += 1
        if self.enable_rebuild: self._maintenance_hook()
        self.delete_count += 1; return True

    def insert_many(self, values, allow_duplicate=False, rebuild=False):
        for v in values: self.insert(v, allow_duplicate=allow_duplicate)
        if rebuild: self._build_junctions()

    def delete_many(self, values, rebuild=False):
        for v in values: self.delete(v)
        if rebuild: self._build_junctions()

    def _maintenance_hook(self):
        if self.local_ops_counter >= self.local_interval:
            bw = int(self.sub_interval * self.stop_crash_local_sub)
            if self.sub_interval > 0 and (self.sub_interval - self.sub_ops_counter) <= bw:
                self.local_ops_counter = max(self.local_interval // 2, 1)
            else:
                self.local_ops_counter = 0
                self.maintenance_step(self.local_interval, self.t_j)
        if self.sub_ops_counter >= self.sub_interval:
            self.sub_ops_counter = 0; self.suboptimal_scan_count += 1
            flagged, hard, seg_ratio = self._scan_suboptimal_segments()
            total = len(self.junctions)
            hard_ratio = len(hard) / max(1, total)
            if hard_ratio >= self.emergency_hard_segment_ratio:
                self.sub_event_before_global_rebuild = 0
                self._perform_global_rebuild(); return
            if (self.sub_event_before_global_rebuild
                    >= self.min_suboptimal_events_before_global
                    and seg_ratio >= self.max_suboptimal_segments):
                self.sub_event_before_global_rebuild = 0
                self._perform_global_rebuild(); return
            fr = len(flagged) / max(1, total)
            if fr >= self.flagged_ratio_limit: self.suboptimal_rebuild_step(flagged)
            elif hard: self.suboptimal_rebuild_step(hard)

    def maintenance_step(self, interval, t_j):
        if not self.junctions: return False
        rebuilt_any = False
        for j in list(self.junctions):
            if not self._live(j): continue
            rebuilt = self.maybe_local_rebuild(j, t_j)
            rebuilt_any = rebuilt_any or rebuilt
        self.local_scan_counter += 1; return rebuilt_any

    def _should_rebuild_and_measure(self, j, t_j):
        left = right = 0; seen = False; cur = j.segment_start
        while cur:
            if cur is j.node: seen = True; right = 1
            elif not seen: left += 1
            else: right += 1
            if cur is j.segment_end: break
            cur = cur.next
        if not seen: return False, 0
        total = left + right
        if total <= 1: return False, total
        return abs(left / total - 0.5) > t_j, total

    def _local_rebuild(self, j, seg_len):
        if seg_len <= 1: return
        plan = self._ps(seg_len); sc_set = set(plan["shortcut_offsets"])
        cur = j.segment_start; idx = 0; new_sc = []; jn = None
        while cur:
            if idx == plan["junction_offset"]: jn = cur
            if idx in sc_set: new_sc.append(cur)
            if cur is j.segment_end: break
            cur = cur.next; idx += 1
        if jn is None: jn = j.segment_start
        sc = [j.segment_start] + new_sc + [j.segment_end]
        if len(sc) < self.K: sc.extend([j.segment_start] * (self.K - len(sc)))
        j.node = jn; j.shortcuts = sc; j.segment_len = seg_len
        self.local_rebuild_events += 1; self.local_rebuild_cost += seg_len

    def maybe_local_rebuild(self, j, t_j):
        should, sl = self._should_rebuild_and_measure(j, t_j)
        if not should: return False
        self._local_rebuild(j, sl); self._last_junction = j; return True

    def _scan_suboptimal_segments(self):
        if not self.junctions: return [], [], 0.0
        soft = []; hard = []
        for i, j in enumerate(self.junctions):
            sl = self._dist(j.segment_start, j.segment_end); j.runtime_len = sl
            drift = abs(sl - self.segment_size) / self.segment_size
            if drift >= self.hard_pct: hard.append(i)
            elif drift >= self.soft_pct: soft.append(i)
        flagged = sorted(set(soft + hard))
        return flagged, hard, len(flagged) / max(1, len(self.junctions))

    def suboptimal_rebuild_step(self, flagged):
        if not self.junctions or not flagged: return 0
        min_seg_len_pct = self.min_seg_len_pct
        seg_size = self.segment_size
        min_region_len = int(seg_size * min_seg_len_pct)
        regions = []; start = prev2 = flagged[0]
        for idx in flagged[1:]:
            if idx == prev2 + 1: prev2 = idx
            else: regions.append((start, prev2)); start = prev2 = idx
        regions.append((start, prev2))
        rebuilt_any = False; aff = 0
        for si, ei in reversed(regions):
            region_start = self.junctions[si].segment_start
            region_end   = self.junctions[ei].segment_end
            L = sum(self.junctions[i].runtime_len for i in range(si, ei + 1))
            if L == 0: continue
            if L < min_region_len:
                if si > 0: si -= 1
                elif ei + 1 < len(self.junctions): ei += 1
                else: continue
                region_start = self.junctions[si].segment_start
                region_end   = self.junctions[ei].segment_end
                _nodes = []; _cur = region_start
                while _cur:
                    _nodes.append(_cur)
                    if _cur is region_end: break
                    _cur = _cur.next
                L = sum(self.junctions[i].runtime_len for i in range(si, ei + 1))
            rem = L; ranges = []
            while rem > 0: sz = min(seg_size, rem); ranges.append(sz); rem -= sz
            plans = [self._ps(s) for s in ranges]
            new_js = []; cur = region_start; re = region_end
            s_i = s_o = 0; ss2 = jn2 = None; sc2 = []; scs2 = set()
            while cur and s_i < len(plans):
                plan = plans[s_i]; sl = plan["seg_len"]
                if s_o == 0: ss2 = cur; jn2 = None; sc2 = [cur]; scs2 = set(plan["shortcut_offsets"])
                if s_o == plan["junction_offset"]: jn2 = cur
                if s_o in scs2: sc2.append(cur)
                if s_o == sl - 1:
                    se2 = cur; sc2.append(cur)
                    if len(sc2) < self.K: sc2.extend([ss2] * (self.K - len(sc2)))
                    if jn2 is None: jn2 = ss2
                    j2 = Junction(jn2); j2.segment_start = ss2; j2.segment_end = se2
                    j2.segment_len = sl; j2.shortcuts = sc2; new_js.append(j2)
                    s_i += 1; s_o = -1; sc2 = []; jn2 = None; scs2 = set()
                if cur is re: break
                cur = cur.next; s_o += 1
            self.junctions[si:ei + 1] = new_js; rebuilt_any = True; aff += len(new_js)
        n = len(self.junctions)
        for i in range(n):
            self.junctions[i].prev_junction = self.junctions[i - 1]
            self.junctions[i].next_junction = self.junctions[(i + 1) % n]
        self._refresh_jid(); self._last_junction = self._last_prev_junction = None
        if rebuilt_any:
            self.suboptimal_rebuild_events += 1
            self.sub_event_before_global_rebuild += 1
            self.suboptimal_segments_cost += aff
        return aff

    def _perform_global_rebuild(self):
        self._build_junctions(); self._last_junction = self._last_prev_junction = None
        self.local_ops_counter = self.sub_ops_counter = 0
        self.global_rebuild_events += 1; self.global_rebuild_cost += self.length


class JunctionLinkedList:
    def __init__(self, segment_size=100, shortcuts_per_junction=3,
                 enable_rebuild=True, local_interval=1000, t_j=0.15,
                 sub_interval=5000, soft_pct=0.25, hard_pct=0.50,
                 flagged_ratio_limit=0.30, min_seg_len_pct=0.25,
                 max_suboptimal_segments=0.3,
                 min_suboptimal_events_before_global=5,
                 emergency_hard_segment_ratio=0.95, stop_crash_local_sub=0.08):
        self._e = _JLIEngine(
            segment_size=segment_size,
            shortcuts_per_junction=shortcuts_per_junction,
            enable_rebuild=enable_rebuild, local_interval=local_interval,
            t_j=t_j, sub_interval=sub_interval, soft_pct=soft_pct,
            hard_pct=hard_pct, flagged_ratio_limit=flagged_ratio_limit,
            min_seg_len_pct=min_seg_len_pct,
            max_suboptimal_segments=max_suboptimal_segments,
            min_suboptimal_events_before_global=min_suboptimal_events_before_global,
            emergency_hard_segment_ratio=emergency_hard_segment_ratio,
            stop_crash_local_sub=stop_crash_local_sub)
    def insert(self, value, payload=None, allow_duplicate=False):
        return self._e.insert(value, payload=payload, allow_duplicate=allow_duplicate)
    def delete(self, value): return self._e.delete(value)
    def search(self, target):
       return self._e.search(target)
      
    def insert_many(self, values, allow_duplicate=False, rebuild=False):
        return self._e.insert_many(values, allow_duplicate=allow_duplicate, rebuild=rebuild)
    def delete_many(self, values, rebuild=False): return self._e.delete_many(values, rebuild=rebuild)
    def build_from_values(self, values, key=lambda x: x):
        return self._e.build_from_values(values, key=key)
    def size(self): return self._e.size()
    def get_metrics(self): return self._e.get_metrics()

JLI = JunctionLinkedList

# ══════════════════════════════════════════════════════════════════════════════
# SKIPLIST
# ══════════════════════════════════════════════════════════════════════════════
import random as _random
_MAX_LEVEL=18;_P=0.25
class _SLNode:
    __slots__=("key","payload","forward")
    def __init__(self,key,level,payload=None): self.key=key;self.payload=payload;self.forward=[None]*(level+1)
class SkipList:
    def __init__(self): self._level=0;self._head=_SLNode(-float("inf"),_MAX_LEVEL);self._size=0
    def _rand_level(self):
        lv=0
        while _random.random()<_P and lv<_MAX_LEVEL: lv+=1
        return lv
    def insert(self,key,payload=None):
        upd=[None]*(_MAX_LEVEL+1);cur=self._head
        for i in range(self._level,-1,-1):
            while cur.forward[i] and cur.forward[i].key<key: cur=cur.forward[i]
            upd[i]=cur
        lv=self._rand_level()
        if lv>self._level:
            for i in range(self._level+1,lv+1): upd[i]=self._head
            self._level=lv
        nd=_SLNode(key,lv,payload=payload)
        for i in range(lv+1): nd.forward[i]=upd[i].forward[i];upd[i].forward[i]=nd
        self._size+=1
    def delete(self,key):
        upd=[None]*(_MAX_LEVEL+1);cur=self._head
        for i in range(self._level,-1,-1):
            while cur.forward[i] and cur.forward[i].key<key: cur=cur.forward[i]
            upd[i]=cur
        tgt=cur.forward[0]
        if not tgt or tgt.key!=key: return False
        for i in range(self._level+1):
            if upd[i].forward[i] is not tgt: break
            upd[i].forward[i]=tgt.forward[i]
        while self._level>0 and not self._head.forward[self._level]: self._level-=1
        self._size-=1;return True
    def search(self,key):
        cur=self._head
        for i in range(self._level,-1,-1):
            while cur.forward[i] and cur.forward[i].key<key: cur=cur.forward[i]
        cur=cur.forward[0];return cur if(cur is not None and cur.key==key) else None
    def __len__(self): return self._size

# ══════════════════════════════════════════════════════════════════════════════
# SHARED INFRASTRUCTURE
# ══════════════════════════════════════════════════════════════════════════════
import csv,math,secrets,statistics,time
from pathlib import Path
from typing import List

OUT_DIR=Path("bench_results");OUT_DIR.mkdir(exist_ok=True)
SEEDS=[42,43,44,45];SEG_SIZE=175;K_SHORTCUTS=6;T_J=0.15;SNAP_EVERY=300

WORKLOADS=[
    ("Read-only",    0.99,0.005),
    ("Search-heavy", 0.80,0.15),
    ("Balanced",     0.34,0.33),
    ("Insert-heavy", 0.15,0.70),
    ("Write-heavy",  0.10,0.50),
    ("Insert-flood", 0.02,0.97),
    ("Delete-flood", 0.02,0.05),
]

def make_payload(): return secrets.token_bytes(4)

def make_jli(**kw):
    p=dict(segment_size=SEG_SIZE,shortcuts_per_junction=K_SHORTCUTS,enable_rebuild=True,
           local_interval=550,t_j=T_J,sub_interval=2000,soft_pct=0.15,hard_pct=0.50,
           flagged_ratio_limit=0.15,min_seg_len_pct=0.25,max_suboptimal_segments=0.70,
           min_suboptimal_events_before_global=5,emergency_hard_segment_ratio=0.90,
           stop_crash_local_sub=0.08)
    p.update(kw);return JunctionLinkedList(**p)

def make_jli_auto(N,**kw):
    seg=max(10,int(math.sqrt(N)*1.01));k=max(4,int(math.sqrt(seg)))
    return make_jli(segment_size=seg,shortcuts_per_junction=k,**kw)

def gen_ops(n,pool,sp,ip):
    ops=[]
    for _ in range(n):
        k=_random.choice(pool);r=_random.random()
        op="search" if r<sp else("insert" if r<sp+ip else"delete")
        ops.append((k,op))
    return ops

def _pct(data,q):
    if not data: return 0.0
    s=sorted(data);return s[min(int(len(s)*q),len(s)-1)]

def window_stats(lats_us):
    if not lats_us: return dict(avg=0.0,p50=0.0,p95=0.0,p99=0.0,max_us=0.0,throughput=0.0)
    total_sec=sum(lats_us)/1e6
    return dict(avg=statistics.mean(lats_us),p50=_pct(lats_us,0.50),
                p95=_pct(lats_us,0.95),p99=_pct(lats_us,0.99),
                max_us=max(lats_us),throughput=len(lats_us)/max(total_sec,1e-9))

def max_p99(records): return max((r.get("p99_us",0) for r in records),default=0)

def open_csv(bench_id,fields):
    path=OUT_DIR/f"bech_{bench_id}.csv"
    fh=open(path,"w",newline="");w=csv.DictWriter(fh,fieldnames=fields);w.writeheader()
    return fh,w,path

# ══════════════════════════════════════════════════════════════════════════════
# INSTRUMENTED WRAPPERS  — record hops + latency in one call
# ══════════════════════════════════════════════════════════════════════════════
class InstrumentedJLI:
    """Counts hops AND records latency per op. flush_window() resets both."""
    def __init__(self,engine):
        self._e=engine
        self.win_sh:List[int]=[];self.win_mh:List[int]=[]
        self.win_lat:List[float]=[]   # microseconds per op (all types)

    def search(self,target):
        t0=time.perf_counter()
        e=self._e;e.search_count+=1;hops=0
        if not e.junctions:
            result=None
        else:
            cj=e._choose_start(target);visited=set();result=None
            while cj not in visited:
                if not e._live(cj): break
                visited.add(cj);ss=cj.segment_start.value;se=cj.segment_end.value
                if not(ss<=target<=se): hops+=1;cj=cj.prev_junction if target<ss else cj.next_junction;continue
                sn=(cj.segment_start if(cj.node and target<cj.node.value) else(cj.node or cj.segment_start))
                valid=[sc for sc in cj.shortcuts if sn.value<sc.value<=target]
                if valid:
                    best=max(valid,key=lambda n:n.value)
                    if best.value>sn.value: hops+=1;sn=best
                cur=sn
                while cur and cur.value<=se:
                    hops+=1
                    if cur.value==target: e._last_junction=cj;result=cur;break
                    if cur.value>target: break
                    cur=cur.next
                break
        self.win_sh.append(hops)
        self.win_lat.append((time.perf_counter()-t0)*1e6)
        return result

    def _mut_hops(self,target):
        e=self._e;hops=0
        if not e.junctions:
            cur=e.head
            while cur and cur.value<target: hops+=1;cur=cur.next
            return hops
        cj=e._choose_start(target)
        if not cj or not e._live(cj):
            cur=e.head
            while cur and cur.value<target: hops+=1;cur=cur.next
            return hops
        visited=set()
        while cj not in visited:
            if not e._live(cj): return hops
            visited.add(cj)
            if cj.segment_start.value<=target<=cj.segment_end.value: break
            hops+=1;cj=cj.prev_junction if target<cj.segment_start.value else cj.next_junction
        sn=cj.segment_start
        for sc in cj.shortcuts:
            if sc.value<target: sn=sc
            else: break
        cur=sn
        while cur and cur.value<target: hops+=1;cur=cur.next
        return hops

    def insert(self,value,payload=None,allow_duplicate=False):
        self.win_mh.append(self._mut_hops(value))
        t0=time.perf_counter()
        r=self._e.insert(value,payload=payload,allow_duplicate=allow_duplicate)
        self.win_lat.append((time.perf_counter()-t0)*1e6);return r

    def delete(self,value):
        self.win_mh.append(self._mut_hops(value))
        t0=time.perf_counter()
        r=self._e.delete(value)
        self.win_lat.append((time.perf_counter()-t0)*1e6);return r

    def get_metrics(self): return self._e.get_metrics()

    def flush_window(self):
        sh=list(self.win_sh);mh=list(self.win_mh);lat=list(self.win_lat)
        self.win_sh.clear();self.win_mh.clear();self.win_lat.clear()
        return sh,mh,lat


class InstrumentedSkipList:
    def __init__(self):
        self._sl=SkipList();self.win_sh:List[int]=[];self.win_mh:List[int]=[];self.win_lat:List[float]=[]
    def _hops(self,key):
        sl=self._sl;hops=0;cur=sl._head
        for i in range(sl._level,-1,-1):
            while cur.forward[i] and cur.forward[i].key<key: cur=cur.forward[i];hops+=1
        return hops
    def search(self,key):
        t0=time.perf_counter();self.win_sh.append(self._hops(key)+1);r=self._sl.search(key)
        self.win_lat.append((time.perf_counter()-t0)*1e6);return r
    def insert(self,key,payload=None):
        self.win_mh.append(self._hops(key));t0=time.perf_counter()
        self._sl.insert(key,payload=payload);self.win_lat.append((time.perf_counter()-t0)*1e6)
    def delete(self,key):
        self.win_mh.append(self._hops(key));t0=time.perf_counter()
        r=self._sl.delete(key);self.win_lat.append((time.perf_counter()-t0)*1e6);return r
    def flush_window(self):
        sh=list(self.win_sh);mh=list(self.win_mh);lat=list(self.win_lat)
        self.win_sh.clear();self.win_mh.clear();self.win_lat.clear()
        return sh,mh,lat

# ══════════════════════════════════════════════════════════════════════════════
# STRUCTURAL INVARIANT CHECKER  (B2 + B3 only — segment count is NOT an invariant)
# ══════════════════════════════════════════════════════════════════════════════
class InvariantError(AssertionError): pass

def check_structural_invariants(engine,label=""):
    e=engine;n=e.length
    result=dict(n=n,seg_count=len(e.junctions),bad_shortcuts=0,bad_junctions=0,B2_ok=True,B3_ok=True)
    if n==0: return result
    bad_sc=0;bad_jn=0
    for j in e.junctions:
        if j.segment_start is None or j.segment_end is None or j.node is None:
            bad_sc+=1;bad_jn+=1;continue
        lo=j.segment_start.value;hi=j.segment_end.value
        # B2: every shortcut non-null and within [lo, hi]
        for sc in j.shortcuts:
            if sc is None or not(lo<=sc.value<=hi): bad_sc+=1
        # B3: junction node within t_j of centre
        left=right=0;seen=False;cur=j.segment_start
        while cur is not None:
            if cur is j.node: seen=True;right=1
            elif not seen: left+=1
            else: right+=1
            if cur is j.segment_end: break
            cur=cur.next
        if seen:
            total=left+right
            if total>1 and abs(left/total-0.5)>e.t_j: bad_jn+=1
    result["bad_shortcuts"]=bad_sc;result["bad_junctions"]=bad_jn
    result["B2_ok"]=bad_sc==0;result["B3_ok"]=bad_jn==0
    violations=[]
    if bad_sc: violations.append(f"B2 {bad_sc} bad shortcuts")
    if bad_jn: violations.append(f"B3 {bad_jn} off-centre junctions")
    if violations: raise InvariantError(f"{label}: "+"; ".join(violations))
    return result

# ══════════════════════════════════════════════════════════════════════════════
# BENCH A — Competitive vs SkipList  (hops + latency + throughput)
# ══════════════════════════════════════════════════════════════════════════════
_A_FIELDS=[
    "bench","run","seed","workload","search_p","insert_p",
    "snapshot","ops_in_window","executed_ops",
    # hops
    "jli_avg_search_hops","jli_avg_mutation_hops","jli_search_hops_cv",
    "sl_avg_search_hops","sl_avg_mutation_hops",
    # latency (us)
    "jli_avg_us","jli_p50_us","jli_p95_us","jli_p99_us","jli_max_us",
    "sl_avg_us","sl_p50_us","sl_p95_us","sl_p99_us","sl_max_us",
    # throughput (ops/sec)
    "jli_throughput","sl_throughput",
    # maintenance (delta this window, denom=mutations only)
    "jli_local_ev_d","jli_sub_ev_d","jli_global_ev_d",
    "jli_maint_per_mutation",
    # normalised
    "jli_hops_per_sqrt_N","sl_hops_per_log2_N","jli_p99_vs_sl_p99",
]

def bench_A(N=30_000,ops=25_000):
    print("\n"+"="*72)
    print(f"  BENCH A — Competitive vs SkipList  (hops + latency + throughput)")
    print(f"  N={N}  ops/workload={ops}  snap every {SNAP_EVERY} ops")
    universe=list(range(N*10));sqrt_N=N**0.5;log2_N=math.log2(N)
    fh,w,path=open_csv("A",_A_FIELDS)
    all_jli_sh:List[float]=[];all_sl_sh:List[float]=[]
    all_p99_ratio:List[float]=[]

    for run_id,seed in enumerate(SEEDS,1):
        _random.seed(seed)
        print(f"\n  Run {run_id} (seed={seed})")
        print(f"  {'Workload':<18} {'JLI sh':>8} {'SL sh':>8} {'JLI p99':>9} {'SL p99':>9} {'p99 ratio':>10} {'JLI tput':>10}")
        init_keys=_random.sample(universe,N)

        for wl,sp,ip in WORKLOADS:
            ops_seq=gen_ops(ops,universe,sp,ip)
            base=make_jli()
            for kk in init_keys: base._e.insert(kk,payload=make_payload())
            ijli=InstrumentedJLI(base._e);prev_m=base.get_metrics().copy()
            isl=InstrumentedSkipList()
            for kk in init_keys: isl.insert(kk,payload=make_payload())
            snap=0;exec_ops=0;wl_jsh:List[float]=[];wl_p99:List[float]=[]
            mut_p=max(1-sp,1e-9)

            for i,(key,op) in enumerate(ops_seq):
                if op=="search": ijli.search(key);isl.search(key)
                elif op=="insert": ijli.insert(key);isl.insert(key)
                else: ijli.delete(key);isl.delete(key)
                exec_ops+=1

                if exec_ops%SNAP_EVERY==0 or i==len(ops_seq)-1:
                    snap+=1
                    jsh_w,jmh_w,jlat_w=ijli.flush_window()
                    ssh_w,smh_w,slat_w=isl.flush_window()
                    cur_m=base.get_metrics()
                    d_loc=cur_m["local_event"]-prev_m.get("local_event",0)
                    d_sub=cur_m["sub_event"]-prev_m.get("sub_event",0)
                    d_glb=cur_m["global_event"]-prev_m.get("global_event",0)
                    n_mut=len(jmh_w)
                    maint_per_mut=(d_loc+d_sub+d_glb)/max(n_mut,1)
                    prev_m=cur_m.copy()
                    js=window_stats(jlat_w);ss=window_stats(slat_w)
                    jsh_avg=statistics.mean(jsh_w) if jsh_w else 0.0
                    ssh_avg=statistics.mean(ssh_w) if ssh_w else 0.0
                    jcv=(statistics.stdev(jsh_w)/max(jsh_avg,1e-9) if len(jsh_w)>=2 else 0.0)
                    jr=jsh_avg/max(sqrt_N,1);sr=ssh_avg/max(log2_N,1)
                    p99r=js["p99"]/max(ss["p99"],1e-9)
                    all_jli_sh.append(jr);all_sl_sh.append(sr);all_p99_ratio.append(p99r)
                    wl_jsh.append(jsh_avg);wl_p99.append(js["p99"])
                    w.writerow({"bench":"A","run":run_id,"seed":seed,"workload":wl,
                        "search_p":sp,"insert_p":ip,"snapshot":snap,
                        "ops_in_window":len(jlat_w),"executed_ops":exec_ops,
                        "jli_avg_search_hops":round(jsh_avg,3),
                        "jli_avg_mutation_hops":round(statistics.mean(jmh_w) if jmh_w else 0,3),
                        "jli_search_hops_cv":round(jcv,4),
                        "sl_avg_search_hops":round(ssh_avg,3),
                        "sl_avg_mutation_hops":round(statistics.mean(smh_w) if smh_w else 0,3),
                        "jli_avg_us":round(js["avg"],3),"jli_p50_us":round(js["p50"],3),
                        "jli_p95_us":round(js["p95"],3),"jli_p99_us":round(js["p99"],3),
                        "jli_max_us":round(js["max_us"],3),
                        "sl_avg_us":round(ss["avg"],3),"sl_p50_us":round(ss["p50"],3),
                        "sl_p95_us":round(ss["p95"],3),"sl_p99_us":round(ss["p99"],3),
                        "sl_max_us":round(ss["max_us"],3),
                        "jli_throughput":round(js["throughput"],1),
                        "sl_throughput":round(ss["throughput"],1),
                        "jli_local_ev_d":d_loc,"jli_sub_ev_d":d_sub,"jli_global_ev_d":d_glb,
                        "jli_maint_per_mutation":round(maint_per_mut,6),
                        "jli_hops_per_sqrt_N":round(jr,4),"sl_hops_per_log2_N":round(sr,4),
                        "jli_p99_vs_sl_p99":round(p99r,4)})

            print(f"  {wl:<18} {statistics.mean(wl_jsh):>8.1f} {'-':>8} "
                  f"{statistics.mean(wl_p99):>9.1f} {'-':>9} {'-':>10} {'-':>10}")

    fh.close()
    print("\n  ASSERTIONS:")
    mj=statistics.mean(all_jli_sh)
    print(f"  [{'PASS' if mj<=3.0 else 'FAIL'}] A1  JLI hops/sqrt_N = {mj:.3f}  (target <=3.0)")
    ms=statistics.mean(all_sl_sh)
    print(f"  [{'PASS' if ms<=3.0 else 'FAIL'}] A2  SL  hops/log2_N = {ms:.3f}  (target <=3.0)")
    cv=statistics.stdev(all_jli_sh)/max(statistics.mean(all_jli_sh),1e-9)
    print(f"  [{'PASS' if cv<0.30 else 'FAIL'}] A3  JLI hop-ratio CV={cv:.4f}  (target <0.30)")
    mr=statistics.mean(all_p99_ratio)
    print(f"  INFO  JLI/SL p99 ratio = {mr:.3f}x  (expected >1 — JLI carries maintenance cost)")
    print(f"  CSV -> {path}")


# ══════════════════════════════════════════════════════════════════════════════
# BENCH 8 — Long-run Latency Drift  (from notebook, 300k ops)
# ══════════════════════════════════════════════════════════════════════════════
_B8_FIELDS=["bench","run","seed","ds","window_num","snapshot","executed_ops",
            "avg_us","p50_us","p95_us","p99_us","max_us","throughput",
            "local_event_t","sub_event_t","global_event_t"]

def bench_8(N=30_000,total_ops=300_000,window=50_000):
    print("\n"+"="*72)
    print(f"  BENCH 8 — Long-run Latency Drift  N={N}  total={total_ops}  window={window}")
    universe=list(range(N*10));fh,w,path=open_csv("8",_B8_FIELDS)
    for run_id,seed in enumerate(SEEDS,1):
        _random.seed(seed)
        print(f"\n  Run {run_id} (seed={seed})")
        print(f"  {'Win':>4}  {'JLI p99':>10}  {'SL p99':>10}  {'global_ev':>12}")
        init_keys=_random.sample(universe,N)
        jli=make_jli()
        sk=SkipList()
        for kk in init_keys: pl=make_payload();jli.insert(kk,payload=pl);sk.insert(kk,payload=pl)
        n_windows=total_ops//window;jli_p99s=[];sk_p99s=[]
        for wn in range(n_windows):
            ops_seq=gen_ops(window,universe,0.33,0.33)
            for ds,ds_name in[(jli,"JLI"),(sk,"SkipList")]:
                win_us=[];snap=0;exec_ops=0
                for i,(key,op) in enumerate(ops_seq):
                    t0=time.perf_counter()
                    if op=="search": ds.search(key)
                    elif op=="insert": ds.insert(key)
                    else: ds.delete(key)
                    win_us.append((time.perf_counter()-t0)*1e6);exec_ops+=1
                    if exec_ops%SNAP_EVERY==0 or i==len(ops_seq)-1:
                        snap+=1;s=window_stats(win_us)
                        m=jli.get_metrics() if ds_name=="JLI" else {"local_event":0,"sub_event":0,"global_event":0}
                        w.writerow({"bench":"8","run":run_id,"seed":seed,"ds":ds_name,
                            "window_num":wn+1,"snapshot":snap,"executed_ops":(wn*window)+exec_ops,
                            "avg_us":round(s["avg"],3),"p50_us":round(s["p50"],3),
                            "p95_us":round(s["p95"],3),"p99_us":round(s["p99"],3),
                            "max_us":round(s["max_us"],3),"throughput":round(s["throughput"],1),
                            "local_event_t":m["local_event"],"sub_event_t":m["sub_event"],
                            "global_event_t":m["global_event"]});win_us=[]
                if ds_name=="JLI": jli_p99s.append(s["p99"])
                else: sk_p99s.append(s["p99"])
            m=jli.get_metrics()
            print(f"  {wn+1:>4}  {jli_p99s[-1]:>10.1f}  {sk_p99s[-1]:>10.1f}  {m['global_event']:>12d}")
        if len(jli_p99s)>=2:
            ratio=jli_p99s[-1]/max(jli_p99s[0],1e-9)
            print(f"  [{'PASS' if ratio<=1.15 else 'FAIL'}] drift first->last: {jli_p99s[0]:.1f}->{jli_p99s[-1]:.1f}us  ratio={ratio:.3f}  (target<=1.15)")
    fh.close();print(f"  CSV -> {path}")

# ══════════════════════════════════════════════════════════════════════════════
# BENCH B — Structural Invariants  (B2 + B3 only)
# ══════════════════════════════════════════════════════════════════════════════
_B_FIELDS=[
    "bench","run","seed","pattern","workload",
    "snapshot","executed_ops","n","seg_count",
    "bad_shortcuts","bad_junctions","B2_ok","B3_ok","cumul_violations",
]
_INSERT_PATTERNS={
    "Random":      lambda n:_random.sample(range(n*10),n),
    "Sorted-asc":  lambda n:list(range(n)),
    "Sorted-desc": lambda n:list(range(n,0,-1)),
    "Localized":   lambda n:[(_random.randint(0,n//10) if _random.random()<0.9
                               else _random.randint(0,n*10)) for _ in range(n)],
    "Clustered":   lambda n:[(_random.randint(0,max(1,n//100)) if _random.random()<0.95
                               else _random.randint(0,n*10)) for _ in range(n)],
}
_B_WORKLOADS=[("Balanced",0.34,0.33),("Insert-flood",0.02,0.97),
              ("Delete-flood",0.02,0.05),("Write-heavy",0.10,0.50)]

def bench_B(N=30_000,ops=30_000):
    print("\n"+"="*72)
    print(f"  BENCH B — Structural Invariants  (B2=shortcut validity, B3=junction centering)")
    print(f"  N={N}  ops/combo={ops}  Segment count is NOT tested — it fluctuates by design")
    universe=list(range(N*10));fh,w,path=open_csv("B",_B_FIELDS)
    grand_v=0;grand_s=0

    for run_id,seed in enumerate(SEEDS,1):
        _random.seed(seed)
        print(f"\n  Run {run_id} (seed={seed})")
        for pat,kf in _INSERT_PATTERNS.items():
            init_keys=kf(N)
            for wl,sp,ip in _B_WORKLOADS:
                jli=make_jli()
                for kk in init_keys: jli._e.insert(kk,payload=make_payload())
                ops_seq=gen_ops(ops,universe,sp,ip)
                snap=0;exec_ops=0;cv=0

                for i,(key,op) in enumerate(ops_seq):
                    if op=="search": jli.search(key)
                    elif op=="insert": jli.insert(key)
                    else: jli.delete(key)
                    exec_ops+=1

                    if exec_ops%SNAP_EVERY==0 or i==len(ops_seq)-1:
                        snap+=1;grand_s+=1;eng=jli._e
                        try:
                            inv=check_structural_invariants(eng,f"r{run_id}/{pat}/{wl}/s{snap}")
                            b2,b3=inv["B2_ok"],inv["B3_ok"]
                            bsc,bjn=inv["bad_shortcuts"],inv["bad_junctions"]
                        except InvariantError as exc:
                            b2=b3=False;bsc=bjn=-1;cv+=1;grand_v+=1
                            print(f"    VIOLATION: {exc}")
                        w.writerow({"bench":"B","run":run_id,"seed":seed,"pattern":pat,
                            "workload":wl,"snapshot":snap,"executed_ops":exec_ops,
                            "n":eng.length,"seg_count":len(eng.junctions),
                            "bad_shortcuts":bsc,"bad_junctions":bjn,
                            "B2_ok":int(b2),"B3_ok":int(b3),"cumul_violations":cv})
                status="PASS" if cv==0 else f"FAIL({cv})"
                print(f"    [{status}] {pat:<14} x {wl:<14}  {snap} snaps")

    fh.close()
    print(f"\n  GRAND ASSERTION:")
    print(f"  [{'PASS' if grand_v==0 else 'FAIL'}] {grand_v} violations / {grand_s} snapshots  (target=0)")
    print(f"  CSV -> {path}")
    return grand_v,grand_s

# ══════════════════════════════════════════════════════════════════════════════
# BENCH C — Workload Independence: show oscillation + bounded ceiling
# ══════════════════════════════════════════════════════════════════════════════
_C_FIELDS=[
    "bench","run","seed","workload","search_p","insert_p",
    "snapshot","ops_in_window","executed_ops",
    # per-window hop stats
    "avg_search_hops","max_search_hops","min_search_hops",
    # rolling ceiling (max of all windows so far for this workload/run)
    "rolling_max_hops","rolling_mean_hops",
    # latency
    "avg_us","p99_us","throughput",
    # maintenance
    "maint_per_mutation",
    # ceiling trend: slope of rolling_max over snapshot index (computed at end, filled per row)
    "ceiling_slope",
]

def bench_C(N=30_000,ops=30_000):
    print("\n"+"="*72)
    print(f"  BENCH C — Workload Independence: hop oscillation + bounded ceiling")
    print(f"  N={N}  ops/profile={ops}  — ceiling should plateau, not grow")
    universe=list(range(N*10));fh,w,path=open_csv("C",_C_FIELDS)

    # We buffer rows per (run,workload) and fill ceiling_slope at the end
    all_rows=[]

    for run_id,seed in enumerate(SEEDS,1):
        _random.seed(seed)
        print(f"\n  Run {run_id} (seed={seed})")
        print(f"  {'Workload':<18} {'snaps':>6} {'mean sh':>9} {'max sh':>9} {'min sh':>9} {'ceil slope':>12}")
        init_keys=_random.sample(universe,N)

        for wl,sp,ip in WORKLOADS:
            ops_seq=gen_ops(ops,universe,sp,ip)
            base=make_jli()
            for kk in init_keys: base._e.insert(kk,payload=make_payload())
            ijli=InstrumentedJLI(base._e);prev_m=base.get_metrics().copy()
            snap=0;exec_ops=0
            rolling_max=0.0;snap_means:List[float]=[];snap_maxes:List[float]=[]
            wl_rows=[]

            for i,(key,op) in enumerate(ops_seq):
                if op=="search": ijli.search(key)
                elif op=="insert": ijli.insert(key)
                else: ijli.delete(key)
                exec_ops+=1

                if exec_ops%SNAP_EVERY==0 or i==len(ops_seq)-1:
                    snap+=1
                    jsh_w,jmh_w,jlat_w=ijli.flush_window()
                    cur_m=base.get_metrics()
                    d_loc=cur_m["local_event"]-prev_m.get("local_event",0)
                    d_sub=cur_m["sub_event"]-prev_m.get("sub_event",0)
                    d_glb=cur_m["global_event"]-prev_m.get("global_event",0)
                    n_mut=len(jmh_w)
                    maint_per_mut=(d_loc+d_sub+d_glb)/max(n_mut,1)
                    prev_m=cur_m.copy()
                    sh_mean=statistics.mean(jsh_w) if jsh_w else 0.0
                    sh_max=max(jsh_w) if jsh_w else 0.0
                    sh_min=min(jsh_w) if jsh_w else 0.0
                    rolling_max=max(rolling_max,sh_mean)
                    snap_means.append(sh_mean);snap_maxes.append(rolling_max)
                    ls=window_stats(jlat_w)
                    row={"bench":"C","run":run_id,"seed":seed,"workload":wl,
                        "search_p":sp,"insert_p":ip,"snapshot":snap,
                        "ops_in_window":len(jlat_w),"executed_ops":exec_ops,
                        "avg_search_hops":round(sh_mean,3),"max_search_hops":sh_max,
                        "min_search_hops":sh_min,"rolling_max_hops":round(rolling_max,3),
                        "rolling_mean_hops":round(statistics.mean(snap_means),3),
                        "avg_us":round(ls["avg"],3),"p99_us":round(ls["p99"],3),
                        "throughput":round(ls["throughput"],1),
                        "maint_per_mutation":round(maint_per_mut,6),"ceiling_slope":0.0}
                    wl_rows.append(row)

            # compute ceiling slope (linear regression of rolling_max vs snap index)
            xs=list(range(1,len(snap_maxes)+1));ys=snap_maxes
            if len(xs)>=2:
                mx=statistics.mean(xs);my=statistics.mean(ys)
                num=sum((x-mx)*(y-my) for x,y in zip(xs,ys))
                den=sum((x-mx)**2 for x in xs)
                slope=num/max(den,1e-9)
            else: slope=0.0
            for r in wl_rows: r["ceiling_slope"]=round(slope,6)
            all_rows.extend(wl_rows)
            print(f"  {wl:<18} {snap:>6} {statistics.mean(snap_means):>9.2f} "
                  f"{max(snap_maxes):>9.2f} {min(snap_means):>9.2f} {slope:>12.4f}")

    for r in all_rows: w.writerow(r)
    fh.close()

    # ASSERTIONS
    print("\n  ASSERTIONS:")
    # C1: ceiling slope <= 0 (not growing) for each workload
    from collections import defaultdict
    wl_slopes=defaultdict(list)
    for r in all_rows: wl_slopes[r["workload"]].append(r["ceiling_slope"])
    positive_slopes=[(wl,slopes[0]) for wl,slopes in wl_slopes.items() if slopes[0]>0.5]
    if not positive_slopes:
        print(f"  [PASS] C1  All workload ceilings plateau (slope <=0.5 hops/snap)")
    else:
        for wl,s in positive_slopes: print(f"  [FAIL] C1  {wl} ceiling slope={s:.4f} (growing)")

    # C2: within same workload, rolling_mean stabilises (last 20% of snaps vs first 20%)
    stable=True
    for r2 in all_rows:
        wl=r2["workload"];break
    # just report ceiling per workload
    print(f"\n  Per-workload hop ceiling (max rolling_max across all runs):")
    wl_ceil=defaultdict(list)
    for r in all_rows: wl_ceil[r["workload"]].append(r["rolling_max_hops"])
    for wl,_,_ in WORKLOADS:
        if wl in wl_ceil:
            c=max(wl_ceil[wl])
            print(f"    {wl:<18}  ceiling={c:.1f} hops")
    print(f"  CSV -> {path}")

# ══════════════════════════════════════════════════════════════════════════════
# BENCH 2 — N-Scaling  (from notebook)
# ══════════════════════════════════════════════════════════════════════════════
_B2_FIELDS=[
    "bench","run","seed","ds","N","snapshot","executed_ops",
    "avg_us","p50_us","p95_us","p99_us","max_us","throughput",
    "local_event_t","sub_event_t","global_event_t",
]

def bench_2(ops=15_000):
    sizes=[5_000,10_000,25_000,50_000,100_000,200_000]
    print("\n"+"="*72)
    print(f"  BENCH 2 — Latency Scaling vs N  (balanced 33/33/33)")
    print(f"  Sizes: {sizes}  ops/size={ops}  make_jli_auto: seg=√N")
    fh,w,path=open_csv("2",_B2_FIELDS)
    scale={}

    for run_id,seed in enumerate(SEEDS,1):
        _random.seed(seed)
        print(f"\n  Run {run_id} (seed={seed})")
        print(f"  {'N':>10}  {'DS':<12}  {'avg_us':>8}  {'p99_us':>8}")
        print("  "+"-"*46)
        for N in sizes:
            universe=list(range(N*10));init_keys=_random.sample(universe,N)
            ops_seq=gen_ops(ops,universe,0.33,0.33)
            # JLI
            jli=make_jli_auto(N)
            for kk in init_keys: jli.insert(kk,payload=make_payload())
            win_us=[];snap=0;exec_ops=0;m_prev=jli.get_metrics().copy()
            for i,(key,op) in enumerate(ops_seq):
                t0=time.perf_counter()
                if op=="search": jli.search(key)
                elif op=="insert": jli.insert(key)
                else: jli.delete(key)
                win_us.append((time.perf_counter()-t0)*1e6);exec_ops+=1
                if exec_ops%SNAP_EVERY==0 or i==len(ops_seq)-1:
                    snap+=1;s=window_stats(win_us);m=jli.get_metrics()
                    w.writerow({"bench":"2","run":run_id,"seed":seed,"ds":"JLI","N":N,
                        "snapshot":snap,"executed_ops":exec_ops,
                        "avg_us":round(s["avg"],3),"p50_us":round(s["p50"],3),
                        "p95_us":round(s["p95"],3),"p99_us":round(s["p99"],3),
                        "max_us":round(s["max_us"],3),"throughput":round(s["throughput"],1),
                        "local_event_t":m["local_event"],"sub_event_t":m["sub_event"],
                        "global_event_t":m["global_event"]})
                    win_us=[]
            p99v=s["p99"];scale.setdefault(N,{}).setdefault("JLI",[]).append(p99v)
            print(f"  {N:>10}  {'JLI':<12}  {s['avg']:>8.1f}  {p99v:>8.1f}")
            # SkipList
            sk=SkipList()
            for kk in init_keys: sk.insert(kk,payload=make_payload())
            win_us=[];snap=0;exec_ops=0
            for i,(key,op) in enumerate(ops_seq):
                t0=time.perf_counter()
                if op=="search": sk.search(key)
                elif op=="insert": sk.insert(key)
                else: sk.delete(key)
                win_us.append((time.perf_counter()-t0)*1e6);exec_ops+=1
                if exec_ops%SNAP_EVERY==0 or i==len(ops_seq)-1:
                    snap+=1;s=window_stats(win_us)
                    w.writerow({"bench":"2","run":run_id,"seed":seed,"ds":"SkipList","N":N,
                        "snapshot":snap,"executed_ops":exec_ops,
                        "avg_us":round(s["avg"],3),"p50_us":round(s["p50"],3),
                        "p95_us":round(s["p95"],3),"p99_us":round(s["p99"],3),
                        "max_us":round(s["max_us"],3),"throughput":round(s["throughput"],1),
                        "local_event_t":0,"sub_event_t":0,"global_event_t":0})
                    win_us=[]
            p99v=s["p99"];scale.setdefault(N,{}).setdefault("SkipList",[]).append(p99v)
            print(f"  {N:>10}  {'SkipList':<12}  {s['avg']:>8.1f}  {p99v:>8.1f}")
            print()

    fh.close()
    print("  ASSERTIONS:")
    for ds_name in["JLI","SkipList"]:
        pts=[(sz,statistics.mean(scale[sz][ds_name])) for sz in sorted(scale) if ds_name in scale[sz]]
        if len(pts)>=2:
            log_n=[math.log(n) for n,_ in pts];log_p=[math.log(max(p,1e-9)) for _,p in pts]
            n2=len(pts);ml=sum(log_n)/n2;mp=sum(log_p)/n2
            num=sum((log_n[i]-ml)*(log_p[i]-mp) for i in range(n2))
            den=sum((log_n[i]-ml)**2 for i in range(n2))
            exp_r=num/den if den else 0
            print(f"  {ds_name} log-log exponent (all N): {exp_r:.3f}")
    if 10_000 in scale and 100_000 in scale:
        j10=statistics.mean(scale[10_000].get("JLI",[1]));j100=statistics.mean(scale[100_000].get("JLI",[1]))
        s10=statistics.mean(scale[10_000].get("SkipList",[1]));s100=statistics.mean(scale[100_000].get("SkipList",[1]))
        ej=math.log(j100/max(j10,1e-9))/math.log(10);es=math.log(s100/max(s10,1e-9))/math.log(10)
        print(f"  [{'PASS' if ej<=0.80 else 'FAIL'}] JLI p99 exponent (10k->100k): {ej:.3f}  (target <=0.80)")
        print(f"  [{'PASS' if ej<=es else 'FAIL'}] JLI exp={ej:.3f} <= SkipList exp={es:.3f}")
    print(f"  CSV -> {path}")
    return scale

# ══════════════════════════════════════════════════════════════════════════════
# BENCH 6 — Config Quality  (from notebook)
# ══════════════════════════════════════════════════════════════════════════════
_B6_FIELDS=["bench","run","seed","config","seg","k","sub_int","local_int",
            "snapshot","executed_ops","avg_us","p95_us","p99_us","max_us","throughput",
            "local_event_t","sub_event_t","global_event_t"]

def bench_6(N=30_000,ops=20_000):
    print("\n"+"="*72)
    print(f"  BENCH 6 — Config Quality  N={N}  ops={ops}")
    configs=[
        ("Tuned(175,k=6)",     175, 6, 2000, 2000),
        ("OldDefault(100,k=3)",100, 3, 5000, 1000),
        ("Undersized_seg=50",   50, 3, 2000, 2000),
        ("Oversized_seg=1500",1500, 3, 2000, 2000),
        ("No_shortcuts(k=2)",  175, 2, 2000, 2000),
        ("Overcuts(k=20)",     175,20, 2000, 2000),
        ("High_sub=40k",       175, 6,40000, 2000),
        ("Low_sub=200",        175, 6,  200, 2000),
        ("High_local=50k",     175, 6, 2000,50000),
        ("Low_local=50",       175, 6, 2000,   50),
    ]
    universe=list(range(N*10));fh,w,path=open_csv("6",_B6_FIELDS)

    for run_id,seed in enumerate(SEEDS,1):
        _random.seed(seed)
        print(f"\n  Run {run_id} (seed={seed})")
        print(f"  {'Config':<26}  {'p99_us':>8}  {'global_ev':>10}")
        init_keys=_random.sample(universe,N)
        ops_seq=gen_ops(ops,universe,0.33,0.33)
        case_p99=[]
        for label,seg,k,sub_int,loc_int in configs:
            jli=make_jli(segment_size=seg,shortcuts_per_junction=k,sub_interval=sub_int,local_interval=loc_int)
            for kk in init_keys: jli.insert(kk,payload=make_payload())
            win_us=[];snap=0;exec_ops=0;last_p99=0
            for i,(key,op) in enumerate(ops_seq):
                t0=time.perf_counter()
                if op=="search": jli.search(key)
                elif op=="insert": jli.insert(key)
                else: jli.delete(key)
                win_us.append((time.perf_counter()-t0)*1e6);exec_ops+=1
                if exec_ops%SNAP_EVERY==0 or i==len(ops_seq)-1:
                    snap+=1;s=window_stats(win_us);m=jli.get_metrics()
                    w.writerow({"bench":"6","run":run_id,"seed":seed,"config":label,
                        "seg":seg,"k":k,"sub_int":sub_int,"local_int":loc_int,
                        "snapshot":snap,"executed_ops":exec_ops,
                        "avg_us":round(s["avg"],3),"p95_us":round(s["p95"],3),
                        "p99_us":round(s["p99"],3),"max_us":round(s["max_us"],3),
                        "throughput":round(s["throughput"],1),
                        "local_event_t":m["local_event"],"sub_event_t":m["sub_event"],
                        "global_event_t":m["global_event"]});win_us=[];last_p99=s["p99"]
            case_p99.append((label,last_p99));m=jli.get_metrics()
            print(f"  {label:<26}  {last_p99:>8.1f}  {m['global_event']:>10d}")
        opt=case_p99[0][1];beats=sum(1 for _,p in case_p99[1:] if p>=opt)
        n_miss=len(case_p99)-1
        print(f"  [{'PASS' if beats>=6 else 'FAIL'}] Tuned p99={opt:.1f}us beats {beats}/{n_miss} misconfigs  (target>=6)")
    fh.close();print(f"  CSV -> {path}")

# ══════════════════════════════════════════════════════════════════════════════
# BENCH D — Is c in seg=c*√N a stable constant across N?
# ══════════════════════════════════════════════════════════════════════════════
_D_FIELDS=["bench","run","seed","N","c","seg_size","snapshot","executed_ops",
           "avg_us","p99_us","throughput","global_event_t",
           "effective_c"]   # actual seg_size/sqrt(N) = c

def bench_D(ops=15_000):
    print("\n"+"="*72)
    print("  BENCH D — c*√N constant: does optimal c stay fixed across N?")
    print("  Sweeps c in [0.5,0.75,1.0,1.01,1.25,1.5,2.0] x N in [5k,25k,100k,200k]")
    sizes=[5_000,25_000,100_000,200_000]
    c_values=[0.50,0.75,1.00,1.01,1.25,1.50,2.00]
    universe_max=200_000*10
    fh,w,path=open_csv("D",_D_FIELDS)
    # best_c[N] = c with lowest mean p99 across seeds
    best_c={}

    for run_id,seed in enumerate(SEEDS,1):
        _random.seed(seed)
        print(f"\n  Run {run_id} (seed={seed})")
        print(f"  {'N':>8}  {'c':>6}  {'seg':>6}  {'p99_us':>9}")
        universe=list(range(universe_max))

        for N in sizes:
            init_keys=_random.sample(universe,N)
            ops_seq=gen_ops(ops,universe,0.33,0.33)
            for c in c_values:
                seg=max(4,int(c*math.sqrt(N)));k=max(4,int(math.sqrt(seg)))
                jli=make_jli(segment_size=seg,shortcuts_per_junction=k)
                for kk in init_keys: jli.insert(kk,payload=make_payload())
                win_us=[];snap=0;exec_ops=0;last_p99=0
                for i,(key,op) in enumerate(ops_seq):
                    t0=time.perf_counter()
                    if op=="search": jli.search(key)
                    elif op=="insert": jli.insert(key)
                    else: jli.delete(key)
                    win_us.append((time.perf_counter()-t0)*1e6);exec_ops+=1
                    if exec_ops%SNAP_EVERY==0 or i==len(ops_seq)-1:
                        snap+=1;s=window_stats(win_us);m=jli.get_metrics()
                        eff_c=seg/math.sqrt(N)
                        w.writerow({"bench":"D","run":run_id,"seed":seed,"N":N,"c":c,
                            "seg_size":seg,"snapshot":snap,"executed_ops":exec_ops,
                            "avg_us":round(s["avg"],3),"p99_us":round(s["p99"],3),
                            "throughput":round(s["throughput"],1),
                            "global_event_t":m["global_event"],"effective_c":round(eff_c,4)})
                        win_us=[];last_p99=s["p99"]
                best_c.setdefault(N,{}).setdefault(c,[]).append(last_p99)
                print(f"  {N:>8}  {c:>6.2f}  {seg:>6}  {last_p99:>9.1f}")

    fh.close()
    print("\n  OPTIMAL c PER N (lowest mean p99 across seeds):")
    print(f"  {'N':>8}  {'best_c':>8}  {'p99_us':>9}  {'2nd_c':>8}  {'2nd_p99':>9}")
    all_best_c=[]
    for N in sizes:
        mean_p99={(c,statistics.mean(v)) for c,v in best_c[N].items()}
        sorted_c=sorted(mean_p99,key=lambda x:x[1])
        bc,bp=sorted_c[0];sc2,sp2=sorted_c[1] if len(sorted_c)>1 else(bc,bp)
        all_best_c.append(bc)
        print(f"  {N:>8}  {bc:>8.2f}  {bp:>9.1f}  {sc2:>8.2f}  {sp2:>9.1f}")
    cv_c=statistics.stdev(all_best_c)/max(statistics.mean(all_best_c),1e-9) if len(all_best_c)>=2 else 0
    print(f"\n  Best-c values: {all_best_c}")
    print(f"  CV of best-c across N: {cv_c:.4f}")
    print(f"  [{'PASS' if cv_c<0.20 else 'FAIL'}] D1  c is stable across N  (CV<0.20)")
    print(f"  CSV -> {path}")

# ══════════════════════════════════════════════════════════════════════════════
# BENCH E — Shortcut scaling: what is k* = f(seg_size)?
# ══════════════════════════════════════════════════════════════════════════════
_E_FIELDS=["bench","run","seed","seg_size","k","snapshot","executed_ops",
           "avg_us","p99_us","throughput","global_event_t",
           "sqrt_seg","log2_seg"]

def bench_E(N=30_000,ops=12_000):
    print("\n"+"="*72)
    print(f"  BENCH E — Shortcut scaling: what is optimal k for each seg_size?")
    print(f"  N={N}  ops={ops}  seg_sizes x k_values swept")
    seg_sizes=[50,100,175,300,500,800]
    k_values=[2,3,4,5,6,8,10,14,20]
    universe=list(range(N*10))
    fh,w,path=open_csv("E",_E_FIELDS)
    # best_k[seg] = k with lowest mean p99 across seeds
    best_k={}

    for run_id,seed in enumerate(SEEDS,1):
        _random.seed(seed)
        print(f"\n  Run {run_id} (seed={seed})")
        print(f"  {'seg':>6}  {'k':>4}  {'p99_us':>9}  {'throughput':>12}")
        init_keys=_random.sample(universe,N)
        ops_seq=gen_ops(ops,universe,0.33,0.33)

        for seg in seg_sizes:
            for k in k_values:
                jli=make_jli(segment_size=seg,shortcuts_per_junction=k)
                for kk in init_keys: jli.insert(kk,payload=make_payload())
                win_us=[];snap=0;exec_ops=0;last_p99=0;last_tput=0
                for i,(key,op) in enumerate(ops_seq):
                    t0=time.perf_counter()
                    if op=="search": jli.search(key)
                    elif op=="insert": jli.insert(key)
                    else: jli.delete(key)
                    win_us.append((time.perf_counter()-t0)*1e6);exec_ops+=1
                    if exec_ops%SNAP_EVERY==0 or i==len(ops_seq)-1:
                        snap+=1;s=window_stats(win_us);m=jli.get_metrics()
                        w.writerow({"bench":"E","run":run_id,"seed":seed,"seg_size":seg,"k":k,
                            "snapshot":snap,"executed_ops":exec_ops,
                            "avg_us":round(s["avg"],3),"p99_us":round(s["p99"],3),
                            "throughput":round(s["throughput"],1),
                            "global_event_t":m["global_event"],
                            "sqrt_seg":round(math.sqrt(seg),3),
                            "log2_seg":round(math.log2(seg),3)})
                        win_us=[];last_p99=s["p99"];last_tput=s["throughput"]
                best_k.setdefault(seg,{}).setdefault(k,[]).append(last_p99)
                print(f"  {seg:>6}  {k:>4}  {last_p99:>9.1f}  {last_tput:>12.0f}")

    fh.close()
    print("\n  OPTIMAL k PER seg_size:")
    print(f"  {'seg':>6}  {'sqrt(seg)':>10}  {'log2(seg)':>10}  {'best_k':>8}  {'p99_us':>9}")
    best_k_vals=[];sqrt_seg_vals=[]
    for seg in seg_sizes:
        mean_p99={(k,statistics.mean(v)) for k,v in best_k[seg].items()}
        bk,bp=min(mean_p99,key=lambda x:x[1])
        best_k_vals.append(bk);sqrt_seg_vals.append(math.sqrt(seg))
        print(f"  {seg:>6}  {math.sqrt(seg):>10.2f}  {math.log2(seg):>10.2f}  {bk:>8}  {bp:>9.1f}")

    # fit k* vs sqrt(seg): log-log regression
    if len(best_k_vals)>=3:
        log_s=[math.log(s) for s in sqrt_seg_vals]
        log_k=[math.log(max(k,1)) for k in best_k_vals]
        ms=statistics.mean(log_s);mk=statistics.mean(log_k)
        num=sum((s-ms)*(k-mk) for s,k in zip(log_s,log_k))
        den=sum((s-ms)**2 for s in log_s)
        exp_sk=num/max(den,1e-9)
        # fit k* vs log2(seg)
        log2_s=[math.log2(s) for s in seg_sizes]
        log_k2=[math.log(max(k,1)) for k in best_k_vals]
        ms2=statistics.mean(log2_s);mk2=statistics.mean(log_k2)
        num2=sum((s-ms2)*(k-mk2) for s,k in zip(log2_s,log_k2))
        den2=sum((s-ms2)**2 for s in log2_s)
        exp_l2=num2/max(den2,1e-9)
        print(f"\n  Log-log fit: best_k ~ sqrt(seg)^{exp_sk:.3f}")
        print(f"  Log-linear : best_k ~ log2(seg)^{exp_l2:.3f}")
        print(f"  Hypothesis: k* ~ sqrt(seg)  →  exponent ≈ 0.5, got {exp_sk:.3f}")
        print(f"  [{'PASS' if abs(exp_sk-0.5)<0.25 else 'UNDETERMINED'}] E1  shortcut scaling consistent with sqrt(seg)")
    print(f"  CSV -> {path}")


In [ ]:
if __name__=="__main__":
    N=30_000;OPS=25_000
    print(f"bench_v2  N={N}  ops={OPS}  snap_every={SNAP_EVERY}  seeds={SEEDS}")
    print(f"Output -> {OUT_DIR.resolve()}\n")

    bench_A(N=N,ops=OPS)
    bench_B(N=N,ops=OPS)
    bench_C(N=N,ops=OPS)
    bench_2(ops=15_000)
    bench_6(N=N,ops=20_000)
    bench_8(N=N,total_ops=300_000,window=50_000)
    bench_D(ops=15_000)
    bench_E(N=N,ops=12_000)

    print("\n"+"="*72+"\n  Complete.")
    for p in sorted(OUT_DIR.glob("bench_*.csv")):
        rows=sum(1 for _ in open(p))-1
        print(f"    {p.name:20s}  {rows:>6} rows  {p.stat().st_size:>8} B")
    print("="*72)

bench_v2  N=30000  ops=25000  snap_every=300  seeds=[42, 43, 44, 45]
Output -> C:\Users\mehal\Downloads\bench_results


  BENCH A — Competitive vs SkipList  (hops + latency + throughput)
  N=30000  ops/workload=25000  snap every 300 ops

  Run 1 (seed=42)
  Workload             JLI sh    SL sh   JLI p99    SL p99  p99 ratio   JLI tput
  Read-only              33.3        -      29.6         -          -          -
  Search-heavy           89.9        -      27.8         -          -          -
  Balanced              119.6        -      31.4         -          -          -
  Insert-heavy          166.3        -      67.2         -          -          -
  Write-heavy           120.7        -      36.9         -          -          -
  Insert-flood          181.8        -      37.0         -          -          -
  Delete-flood           61.9        -      19.1         -          -          -

  Run 2 (seed=43)
  Workload             JLI sh    SL sh   JLI p99    SL p99  p99 ratio   JLI t